# 04 – Analysis
Statistical checks on the key comparisons in the metric views, for the Data Science audience. Tests only the comparisons presented as findings. All results are associations: groups were not randomized.

## 1. Churn by billing cycle (June–August 2026)
Question: do monthly-billed customers really churn more than annual and lifetime customers, or could the gap be chance?

In [0]:
%python
from scipy.stats import fisher_exact, chi2_contingency

churn = spark.sql("""
    SELECT billing_cycle, starting_customers, churned_customers, churn_rate
    FROM workspace.scribd_exercise.metrics_churn
    WHERE window_end = DATE'2026-08-31'
      AND business_id = 'All businesses'
      AND billing_cycle <> 'All billing cycles'
""").toPandas().set_index('billing_cycle')
display(churn)

# Table of churned vs. stayed for each billing cycle
def counts(cycle):
    churned = int(churn.loc[cycle, 'churned_customers'])
    stayed = int(churn.loc[cycle, 'starting_customers']) - churned
    return [churned, stayed]

# All three billing cycles at once: do churn rates differ at all?
chi2, p_all, _, _ = chi2_contingency([counts('monthly'), counts('annual'), counts('lifetime_deal_fake')])
print(f"All three billing cycles (chi-square): p = {p_all:.6f}")

# Headline pair: monthly vs. lifetime
odds_ratio, p_pair = fisher_exact([counts('monthly'), counts('lifetime_deal_fake')])
print(f"Monthly vs. lifetime (Fisher's exact): odds ratio = {odds_ratio:.1f}, p = {p_pair:.6f}")

starting_customers,churned_customers,churn_rate
123,12,0.0976
81,22,0.2716
83,4,0.0482


All three billing cycles (chi-square): p = 0.000043
Monthly vs. lifetime (Fisher's exact): odds ratio = 7.4, p = 0.000081


**Result:** 
- Churn differs by billing cycle well beyond chance (chi-square across the three cycles, p < 0.001)
- Monthly-billed customers had about 7 times the odds of churning compared with lifetime customers (Fisher's exact test, odds ratio 7.4, p < 0.001): 27.2% vs. 4.8% over June–August 2026.

**Caveat:** association, not cause. Customers weren't randomly assigned to billing cycles, and the data doesn't show how a customer gets a lifetime deal (chosen, offered as a promotion or retention offer, or bought through a partner), so lifetime customers may differ from others in ways we can't see. Whether moving customers to annual or lifetime plans reduces churn would need an experiment.

## 2. Churn by business (June–August 2026)
Question: BIZ_1 churns most (15.7%) and BIZ_3 least (11.1%). Is that difference real, or chance?

In [0]:
%python
churn_biz = spark.sql("""
    SELECT business_id, starting_customers, churned_customers, churn_rate,
           churn_rate_ci_low, churn_rate_ci_high
    FROM workspace.scribd_exercise.metrics_churn
    WHERE window_end = DATE'2026-08-31'
      AND business_id <> 'All businesses'
      AND billing_cycle = 'All billing cycles'
    ORDER BY business_id
""").toPandas()
display(churn_biz)

# Churned vs. stayed for each business
table = [[int(r.churned_customers), int(r.starting_customers - r.churned_customers)]
         for r in churn_biz.itertuples()]

chi2, p_biz, _, _ = chi2_contingency(table)
print(f"Churn across the three businesses (chi-square): p = {p_biz:.3f}")

business_id,starting_customers,churned_customers,churn_rate,churn_rate_ci_low,churn_rate_ci_high
BIZ_1,89,14,0.1573,0.0961,0.2469
BIZ_2,108,14,0.1296,0.0788,0.2059
BIZ_3,90,10,0.1111,0.0615,0.1926


Churn across the three businesses (chi-square): p = 0.656


**Result:** 
- Churn does not differ clearly between businesses (chi-square, p = 0.66). 
- BIZ_1's 15.7% vs. BIZ_3's 11.1% is within what chance would produce with about 90–110 customers per business, and their confidence intervals overlap. 
- No business should be singled out for a churn problem on this evidence.

## 3. Catalog reach by content age (August 2026)
Question: older content (366+ days) was seen less often (66%) than newer content (new 100%, recent 95%). Is the gap real?
New and recent are combined, since every new item was reached (22 of 22), which leaves too few unreached items to compare separately.

In [0]:
%python
reach = spark.sql("""
    SELECT dimension_value AS age_band, live_items, reached_items, catalog_reach,
           catalog_reach_ci_low, catalog_reach_ci_high
    FROM workspace.scribd_exercise.metrics_catalog_reach
    WHERE month_start = DATE'2026-08-01'
      AND business_id = 'All businesses'
      AND dimension = 'content age'
""").toPandas().set_index('age_band')
display(reach)

older_reached = int(reach.loc['Older (366+ days)', 'reached_items'])
older_unreached = int(reach.loc['Older (366+ days)', 'live_items']) - older_reached

newer = reach.drop('Older (366+ days)')
newer_reached = int(newer['reached_items'].sum())
newer_unreached = int(newer['live_items'].sum()) - newer_reached

odds_ratio, p_reach = fisher_exact([[older_reached, older_unreached],
                                    [newer_reached, newer_unreached]])
print(f"Older: {older_reached} of {older_reached + older_unreached} reached; "
      f"newer (0-365 days): {newer_reached} of {newer_reached + newer_unreached} reached")
print(f"Older vs. newer (Fisher's exact): odds ratio = {odds_ratio:.2f}, p = {p_reach:.2e}")

live_items,reached_items,catalog_reach,catalog_reach_ci_low,catalog_reach_ci_high
65,62,0.9538,0.8729,0.9842
199,132,0.6633,0.5951,0.7253
22,22,1.0,0.8513,1.0


Older: 132 of 199 reached; newer (0-365 days): 84 of 87 reached
Older vs. newer (Fisher's exact): odds ratio = 0.07, p = 1.88e-09


**Result:** 
- Older content was far less likely to be seen in August 2026: 132 of 199 older items (66%) vs. 84 of 87 newer items (97%) had at least one content event (Fisher's exact test, odds ratio 0.07, p < 0.001). 
- Older items had about 1/14 the odds of being seen.

**Caveat:** association, not cause. Older content may get less attention because of discovery (how it's surfaced) or because demand naturally fades with age; this data can't separate the two. A test that resurfaces older content (e.g., in recommendations) would tell them apart.

## 4. Experiment readiness
Question: with today's customer base, what size of churn change could an A/B test detect?
Assumptions: two equal arms, 5% significance (two-sided), 80% power, baseline = 3-month churn for June–August 2026.

In [0]:
%python
from math import sqrt, ceil
from scipy.stats import norm

base = spark.sql("""
    SELECT starting_customers, churn_rate
    FROM workspace.scribd_exercise.metrics_churn
    WHERE window_end = DATE'2026-08-31'
      AND business_id = 'All businesses'
      AND billing_cycle = 'All billing cycles'
""").first()

p = float(base.churn_rate)
customers = int(base.starting_customers)
per_arm = customers // 2
z = norm.ppf(1 - 0.05 / 2) + norm.ppf(0.80)

# Smallest detectable change with today's base
mde = z * sqrt(2 * p * (1 - p) / per_arm)
print(f"Baseline 3-month churn: {p:.1%}, {customers} customers ({per_arm} per arm)")
print(f"Minimum detectable effect: about {mde * 100:.1f} percentage points")

# Customers needed per arm to detect a 3-point drop
target = p - 0.03
p_avg = (p + target) / 2
n_needed = ((norm.ppf(0.975) * sqrt(2 * p_avg * (1 - p_avg))
             + norm.ppf(0.80) * sqrt(p * (1 - p) + target * (1 - target))) ** 2) / (p - target) ** 2
print(f"To detect a 3-point drop ({p:.1%} -> {target:.1%}): about {ceil(n_needed):,} customers per arm, "
      f"{ceil(n_needed) * 2 / customers:.0f}x the 287 cusotmers who were paying at the start of the June-August window")

Baseline 3-month churn: 13.2%, 287 customers (143 per arm)
Minimum detectable effect: about 11.2 percentage points
To detect a 3-point drop (13.2% -> 10.2%): about 1,807 customers per arm, 13x the 287 cusotmers who were paying at the start of the June-August window
